# Lab 13 - PyTorch and GPU

In [1]:
import torch
import torch.nn as nn
import numpy as np
import math
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'

device

'cuda'

## Ex.1) Training with Stream-based Prefetching

Goal:

Overlap **CPU data loading + H2D transfer** with **GPU compute** by:
- Using `DataLoader(pin_memory=True)` so batches arrive in pinned host memory.
- Using a **dedicated copy stream** to enqueue `to(device, non_blocking=True)`.
- Using `wait_stream` to ensure the default/compute stream waits only when needed

In [9]:
# We want to build a CUDAPrefetcher class that wraps a DataLoader by parallelizing Data Loading and GPU Compute:

class CUDAPrefetcher:
    """
    Prefetches the next batch to GPU on a dedicated CUDA stream.
    Assumes DataLoader(pin_memory=True) so H2D can be async.
    """
    def __init__(self, data_loader, device):
        self.loader = iter(data_loader)
        self.device = device
        self.copy_stream = torch.cuda.Stream(device=device)
        self.next_batch = None
        self._preload()
        
    def _to_device(self, batch):
        """Transfers batch to self.device (handles 1-deep nested structures)"""
        if isinstance(batch, (tuple, list)):
            return [t.to(self.device, non_blocking=True) for t in batch]
        return batch.to(self.device, non_blocking=True)
        
    def _preload(self):
        """Loads self.next_batch"""
        try:
            batch = next(self.loader)
            with torch.cuda.stream(self.copy_stream):
                self.next_batch = self._to_device(batch)
        except StopIteration:
            self.next_batch = None
            
    def __iter__(self):
        return self

    def __next__(self):
        if self.next_batch is None:
            raise StopIteration
    
        # Synchronize Default Stream with copy_stream!
        torch.cuda.current_stream(self.device).wait_stream(self.copy_stream)
        
        batch = self.next_batch
        self._preload()
        return batch

In [34]:
# Datasets and Data Loader

transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    ]
)


data_root = "../../data"

train_ds = datasets.CIFAR10(
    root=data_root, train=True, download=True, transform=transform_train
)
test_ds = datasets.CIFAR10(
    root=data_root, train=False, download=True, transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    drop_last=True,
)

test_loader = torch.utils.data.DataLoader(
    test_ds,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)

TRAIN_SIZE = len(train_ds)
EVAL_SIZE = len(test_ds)

In [39]:
class Model(nn.Module):
    
    def __init__(self, num_channels, num_classes): # Inputs are [3x32x32]
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=num_channels, out_channels=32, kernel_size=3, stride=2, padding=1), # 16x16
            nn.ReLU(),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=2, padding=1), # 8x8
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=2, padding=1), # 4x4
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(in_features=128 * 4 * 4, out_features=256),
            nn.Dropout(),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=num_classes)
        )
        
    def forward(self, x):
        return self.layers(x)
    
model = Model(num_channels=3, num_classes=10).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1E-3)
loss_function = nn.CrossEntropyLoss()


EPOCHS = 50

for epoch in range(EPOCHS):
    train_loss = 0.0
    eval_loss = 0.0
    
    train_correct = 0
    eval_correct = 0
    
    train_prefetcher = CUDAPrefetcher(train_loader, device)
    eval_prefetcher = CUDAPrefetcher(test_loader, device)
        
    model.train()
    for batch_idx, (inputs, targets) in enumerate(train_prefetcher):
        outputs = model(inputs)
        output_preds = outputs.argmax(dim=1)

        optimizer.zero_grad()
        loss = loss_function(outputs, targets)
        train_loss += loss.item() * inputs.size(0)
        train_correct += (output_preds == targets).sum().item()
        
        loss.backward()
        optimizer.step()
        
    model.eval()
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(eval_prefetcher):
            outputs = model(inputs)
            output_preds = outputs.argmax(dim=1)

            loss = loss_function(outputs, targets)
            eval_loss += loss.item() * inputs.size(0)
            eval_correct += (output_preds == targets).sum().item()
            
    if epoch % 10 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch: {epoch} | Train Loss: {train_loss / TRAIN_SIZE} | Eval Loss: {eval_loss / EVAL_SIZE} | Train Acc: {train_correct / TRAIN_SIZE} | Eval Acc: {eval_correct / EVAL_SIZE}")

Epoch: 0 | Train Loss: 1.7440194250488281 | Eval Loss: 1.422639060974121 | Train Acc: 0.35764 | Eval Acc: 0.4784
Epoch: 10 | Train Loss: 0.9480133306884766 | Eval Loss: 0.8394174858093262 | Train Acc: 0.66406 | Eval Acc: 0.7093
Epoch: 20 | Train Loss: 0.8056001405334473 | Eval Loss: 0.726436505317688 | Train Acc: 0.71612 | Eval Acc: 0.7486
Epoch: 30 | Train Loss: 0.7371810163879394 | Eval Loss: 0.6749140908241272 | Train Acc: 0.74206 | Eval Acc: 0.7691
Epoch: 40 | Train Loss: 0.7003408895874024 | Eval Loss: 0.6642875151634217 | Train Acc: 0.75432 | Eval Acc: 0.7737
Epoch: 49 | Train Loss: 0.6668576972961425 | Eval Loss: 0.6175074698448181 | Train Acc: 0.76718 | Eval Acc: 0.7855


In [10]:
image, target_class = train_ds[0]
image.shape

torch.Size([3, 32, 32])

In [14]:
class CifarCNN(nn.Module):
    
    def __init__(self, num_channels, num_classes):
        super(CifarCNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=num_channels, out_channels=64, kernel_size=3, stride=1, padding=1), # [64x32x32] -> [64x32x32]
            nn.ReLU(),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1, padding=1), # [64x32x32] -> [64x32x32]
            nn.ReLU(),
            nn.MaxPool2d(2), # [64x32x32] -> [64x16x16]
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1, padding=1), # [64x16x16] -> [128x16x16]
            nn.ReLU(),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding=1), # [128x16x16] -> [128x16x16]
            nn.ReLU(),
            nn.MaxPool2d(2), # [128x16x16] -> [128x8x8]
            nn.Flatten(), # [128x8x8] -> [128 * 8 * 8]
            nn.Linear(in_features=128 * 8 * 8, out_features=256),
            nn.ReLU(),
            nn.Linear(in_features=256, out_features=num_classes)
        )
        
    def forward(self, x):
        return self.layers(x)
    
model = CifarCNN(3, 10).to(device)

In [32]:
LR = 3E-4
EPOCHS = 25

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_function = nn.CrossEntropyLoss()

train_losses = list()
train_accuracies = list()
test_losses = list()
test_accuracies = list()

for epoch in range(EPOCHS):
    current_train_loss = 0.0
    train_total, train_correct = 0, 0
    
    current_test_loss = 0.0
    test_total, test_correct = 0, 0
    
    train_prefetcher = CUDAPrefetcher(train_loader, device)
    
    # Train loop
    model.train()
    for i, batch in enumerate(train_prefetcher):
        optimizer.zero_grad()
        images, target_classes = batch
        images = images.to(device)
        target_classes = target_classes.to(device)
        
        pred_logits = model(images)
        
        loss = loss_function(pred_logits, target_classes)
        
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            current_train_loss += loss.item()
            pred_classes = torch.argmax(pred_logits, dim=1)
            train_total += pred_classes.size(0)
            train_correct += (pred_classes == target_classes).sum().item()
            
    # Test loop
    model.eval()
    with torch.no_grad():
        for i, batch in enumerate(test_loader):
            images, target_classes = batch
            images = images.to(device)
            target_classes = target_classes.to(device)
            
            pred_logits = model(images)
            loss = loss_function(pred_logits, target_classes)

            current_test_loss += loss.item()
            pred_classes = torch.argmax(pred_logits, dim=1)
            test_total += pred_classes.size(0)
            test_correct += (pred_classes == target_classes).sum().item()
        
    train_losses.append(current_train_loss)
    train_accuracies.append(train_correct / train_total)
    test_losses.append(current_test_loss)
    test_accuracies.append(test_correct / test_total)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d} | "
              f"Train Loss {current_train_loss:5.5f} and Accuracy {(train_correct / train_total):3.3f} | "
              f"Test Loss {current_test_loss:5.5f} and Accuracy {(test_correct / test_total):3.3f}"
              )

Epoch    0 | Train Loss 226.78343 and Accuracy 0.794 | Test Loss 51.57297 and Accuracy 0.778
Epoch   10 | Train Loss 198.92657 and Accuracy 0.820 | Test Loss 50.08721 and Accuracy 0.788
Epoch   20 | Train Loss 184.77697 and Accuracy 0.831 | Test Loss 49.50725 and Accuracy 0.793
